# 03 - Yogyakarta Anomaly Evaluation and Alerts

This notebook scores the trained LSTM Autoencoder, applies physical rainfall and wind rules, and exports dissertation-ready anomaly tables.

In [ ]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

sns.set_theme(style="whitegrid")

PROJECT_ROOT = Path("..").resolve()
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed" / "yogyakarta_weather_features.csv"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
REPORT_DIR = PROJECT_ROOT / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

model = tf.keras.models.load_model(ARTIFACT_DIR / "model_lstm_autoencoder.keras")
scaler = joblib.load(ARTIFACT_DIR / "scaler.pkl")
thresholds = json.loads((ARTIFACT_DIR / "threshold.json").read_text(encoding="utf-8"))
feature_config = json.loads((ARTIFACT_DIR / "feature_config.json").read_text(encoding="utf-8"))

model_features = feature_config["model_features"]
feature_weights = feature_config["feature_weights"]
SEQUENCE_LENGTH = int(feature_config["sequence_length"])


In [ ]:
data = pd.read_csv(PROCESSED_PATH, parse_dates=["date"], dtype={"station_id": "string"})
data = data.sort_values(["station_id", "date"]).reset_index(drop=True)

ordered_dates = np.array(sorted(data["date"].unique()))
n_dates = len(ordered_dates)
test_size = int(round(n_dates * 0.15))
validation_size = int(round(n_dates * 0.15))
train_end = n_dates - validation_size - test_size
validation_end = n_dates - test_size

training_threshold_df = data[data["date"].isin(ordered_dates[:train_end])].copy()
evaluation_dates = ordered_dates[train_end:]
evaluation_df = data[data["date"].isin(evaluation_dates)].copy()
evaluation_scaled = evaluation_df.copy()
evaluation_scaled[model_features] = scaler.transform(evaluation_scaled[model_features])


def build_sequences(feature_df: pd.DataFrame, metadata_df: pd.DataFrame, feature_names: list, sequence_length: int):
    x_values = []
    metadata_rows = []
    skipped_windows = 0

    sorted_features = feature_df.sort_values(["station_id", "date"])
    sorted_metadata = metadata_df.sort_values(["station_id", "date"])

    for station_id, group in sorted_features.groupby("station_id", sort=False):
        g = group.sort_values("date").reset_index(drop=True)
        meta_g = sorted_metadata[sorted_metadata["station_id"] == station_id].sort_values("date").reset_index(drop=True)
        if not g["date"].equals(meta_g["date"]):
            raise ValueError(f"Scaled and raw metadata dates do not match for station {station_id}")
        values = g[feature_names].to_numpy(dtype=np.float32)

        for end_idx in range(sequence_length - 1, len(g)):
            start_idx = end_idx - sequence_length + 1
            window_dates = g.loc[start_idx:end_idx, "date"]
            day_differences = window_dates.diff().dt.days.iloc[1:]

            if not (day_differences == 1).all():
                skipped_windows += 1
                continue

            x_values.append(values[start_idx:end_idx + 1])
            metadata_rows.append(meta_g.iloc[end_idx].to_dict())

    print("Skipped non-consecutive date windows:", skipped_windows)

    if not x_values:
        return np.empty((0, sequence_length, len(feature_names)), dtype=np.float32), pd.DataFrame(metadata_rows)

    return np.stack(x_values).astype(np.float32), pd.DataFrame(metadata_rows)


x_eval, eval_meta = build_sequences(evaluation_scaled, evaluation_df, model_features, SEQUENCE_LENGTH)
eval_pred = model.predict(x_eval, verbose=0)

print("Evaluation sequences:", x_eval.shape)
display(eval_meta.head())


In [ ]:
def weight_vector(feature_names: list) -> np.ndarray:
    weights = np.array([feature_weights.get(name, 1.0) for name in feature_names], dtype=np.float32)
    return weights / weights.mean()


def weighted_reconstruction_error(x_true: np.ndarray, x_pred: np.ndarray, feature_names: list) -> np.ndarray:
    weights = weight_vector(feature_names).reshape(1, 1, -1)
    squared_error = np.square(x_true - x_pred)
    return np.mean(squared_error * weights, axis=(1, 2))


def anomaly_level(score: float, thresholds: dict) -> str:
    if score > thresholds["p995"]:
        return "AWAS"
    if score > thresholds["p99"]:
        return "SIAGA"
    if score > thresholds["p95"]:
        return "WASPADA"
    return "NORMAL"


scores = weighted_reconstruction_error(x_eval, eval_pred, model_features)
eval_meta["anomaly_score"] = scores
eval_meta["status"] = [anomaly_level(float(score), thresholds) for score in scores]

anomaly_scores_path = REPORT_DIR / "anomaly_scores.csv"
eval_meta.to_csv(anomaly_scores_path, index=False)

display(eval_meta[["date", "station_id", "region_name", "anomaly_score", "status"]].head())
print("Saved:", anomaly_scores_path)


In [ ]:
rain_daily_threshold = float(training_threshold_df["RR"].quantile(0.95))
rain_3d_threshold = float(training_threshold_df["rain_3d"].quantile(0.95))
wind_max_threshold = float(training_threshold_df["ff_x"].quantile(0.95))
wind_avg_threshold = float(training_threshold_df["ff_avg"].quantile(0.95))

print("Rain daily threshold:", rain_daily_threshold)
print("Rain 3-day threshold:", rain_3d_threshold)
print("Wind max threshold:", wind_max_threshold)
print("Wind avg threshold:", wind_avg_threshold)


def build_alert(row: pd.Series) -> dict:
    rain_alert = bool(row["RR"] >= rain_daily_threshold or row["rain_3d"] >= rain_3d_threshold)
    wind_alert = bool(row["ff_x"] >= wind_max_threshold or row["ff_avg"] >= wind_avg_threshold)

    if row["status"] == "NORMAL":
        alert_type = "NORMAL"
    elif rain_alert and wind_alert:
        alert_type = "CURAH_HUJAN_EKSTREM_DAN_ANGIN_KENCANG"
    elif rain_alert:
        alert_type = "CURAH_HUJAN_EKSTREM"
    elif wind_alert:
        alert_type = "ANGIN_KENCANG"
    else:
        alert_type = "ANOMALI_CUACA"

    triggers = []
    if row["status"] != "NORMAL":
        triggers.append(f"Anomaly score above {row['status']} threshold")
        if rain_alert:
            triggers.append("Rainfall above local percentile threshold")
        if wind_alert:
            triggers.append("Wind above local percentile threshold")

    return {
        "date": pd.Timestamp(row["date"]).date().isoformat(),
        "station_id": str(row["station_id"]),
        "region_name": row["region_name"],
        "status": row["status"],
        "alert_type": alert_type,
        "source": "LSTM_AUTOENCODER",
        "anomaly_score": float(row["anomaly_score"]),
        "threshold_p95": float(thresholds["p95"]),
        "threshold_p99": float(thresholds["p99"]),
        "threshold_p995": float(thresholds["p995"]),
        "rr_mm": float(row["RR"]),
        "rain_3d_mm": float(row["rain_3d"]),
        "rain_7d_mm": float(row["rain_7d"]),
        "ff_x": float(row["ff_x"]),
        "ff_avg": float(row["ff_avg"]),
        "triggers": "; ".join(triggers),
    }


alerts = pd.DataFrame([build_alert(row) for _, row in eval_meta.iterrows()])
alerts_path = REPORT_DIR / "alerts.csv"
alerts.to_csv(alerts_path, index=False)
display(alerts.head())
print("Saved:", alerts_path)


In [ ]:
top_anomalies = alerts.sort_values("anomaly_score", ascending=False).head(50).reset_index(drop=True)
station_counts = alerts.groupby(["station_id", "status", "alert_type"]).size().reset_index(name="count")

top_anomalies.to_csv(REPORT_DIR / "top_anomalies.csv", index=False)
station_counts.to_csv(REPORT_DIR / "station_anomaly_counts.csv", index=False)

display(top_anomalies.head(20))
display(station_counts)


In [ ]:
threshold_sensitivity = (
    alerts["status"]
    .value_counts()
    .reindex(["NORMAL", "WASPADA", "SIAGA", "AWAS"], fill_value=0)
    .rename_axis("status")
    .reset_index(name="count")
)
threshold_sensitivity.to_csv(REPORT_DIR / "threshold_sensitivity.csv", index=False)
display(threshold_sensitivity)


In [ ]:
plt.figure(figsize=(12, 6))
sns.histplot(alerts["anomaly_score"], bins=50, kde=True)
plt.axvline(thresholds["p95"], color="orange", linestyle="--", label="P95")
plt.axvline(thresholds["p99"], color="red", linestyle="--", label="P99")
plt.axvline(thresholds["p995"], color="purple", linestyle="--", label="P99.5")
plt.title("Weighted Reconstruction Error Distribution")
plt.xlabel("Anomaly Score")
plt.ylabel("Window Count")
plt.legend()
plt.tight_layout()
plt.savefig(REPORT_DIR / "reconstruction_error_distribution.png", dpi=160)
plt.show()


In [ ]:
plot_data = alerts.copy()
plot_data["date"] = pd.to_datetime(plot_data["date"])

plt.figure(figsize=(14, 6))
for station_id, group in plot_data.groupby("station_id"):
    plt.plot(group["date"], group["anomaly_score"], label=station_id, alpha=0.8)

plt.axhline(thresholds["p95"], color="orange", linestyle="--", label="P95")
plt.axhline(thresholds["p99"], color="red", linestyle="--", label="P99")
plt.axhline(thresholds["p995"], color="purple", linestyle="--", label="P99.5")
plt.title("Yogyakarta LSTM Autoencoder Anomaly Timeline")
plt.xlabel("Date")
plt.ylabel("Anomaly Score")
plt.legend()
plt.tight_layout()
plt.show()
